# SER · Session 2 — Training (GPU)

**Attach:** the four corpora, **plus** the `ser-feature-cache` dataset produced
by notebook 01, **plus** (on a resumed run) the previous version's output so
finished runs carry forward.

**Accelerator: GPU P100 or T4 x2.**

~6–9 GPU-hours for all six configurations. The runner is **resumable** — it
skips any config that already has `test_metrics.json` — so it is safe to split
across several sessions.

In [ ]:
import glob
import json
import os
import shutil
import sys
import time

import tensorflow as tf
import keras

gpus = tf.config.list_physical_devices("GPU")
print("TF", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", gpus)
assert gpus, "No GPU. Set Settings -> Accelerator -> GPU before running."

In [ ]:
REPO = "https://github.com/Eldorado5002/ser.git"

if not os.path.exists("/kaggle/working/ser"):
    !git clone -q {REPO} /kaggle/working/ser

sys.path.insert(0, "/kaggle/working/ser")
os.chdir("/kaggle/working/ser")
!git log --oneline -1

In [ ]:
# PINNED path. Both notebooks MUST use this identical string: the feature
# cache fingerprint hashes ABSOLUTE file paths, so a different root silently
# invalidates every cache and re-extracts features on GPU time.
DATA_ROOT = "/kaggle/working/ser/data"

SOURCES = {
    "RAVDESS": "/kaggle/input/ravdess-emotional-speech-audio/audio_speech_actors_01-24",
    "TESS":    "/kaggle/input/toronto-emotional-speech-set-tess/TESS Toronto emotional speech set data",
    "SAVEE":   "/kaggle/input/surrey-audiovisual-expressed-emotion-savee/ALL",
    "CREMA-D": "/kaggle/input/cremad/AudioWAV",
}

os.makedirs(DATA_ROOT, exist_ok=True)
for name, src in SOURCES.items():
    dst = os.path.join(DATA_ROOT, name)
    assert os.path.isdir(src), (
        f"MISSING INPUT: {src}\n"
        f"Attach the dataset, then check the mount path under /kaggle/input/.")
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.exists(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f"{name:9s} -> {src}")

In [ ]:
import config

config.CACHE_DIR = "/kaggle/working/features_cache"
config.RUNS_DIR = "/kaggle/working/runs"
os.makedirs(config.CACHE_DIR, exist_ok=True)
os.makedirs(config.RUNS_DIR, exist_ok=True)

# Copy rather than symlink: build_feature_matrix writes into CACHE_DIR.
staged = 0
for pattern in ("/kaggle/input/*/*.npz", "/kaggle/input/*/*/*.npz"):
    for f in glob.glob(pattern):
        shutil.copy(f, config.CACHE_DIR)
        staged += 1
print(f"staged {staged} cache files")

# Resume: bring forward completed runs from a previous session's output.
for prior in glob.glob("/kaggle/input/*/runs/*/test_metrics.json"):
    tag_dir = os.path.dirname(prior)
    dst = os.path.join(config.RUNS_DIR, os.path.basename(tag_dir))
    if not os.path.exists(dst):
        shutil.copytree(tag_dir, dst)

print("caches:", [os.path.basename(f)
                  for f in sorted(glob.glob(f"{config.CACHE_DIR}/*.npz"))])
print("already finished:", sorted(
    os.path.basename(os.path.dirname(p))
    for p in glob.glob(f"{config.RUNS_DIR}/*/test_metrics.json")))

In [ ]:
from augmentation import plan_augmentation
from data_loader import build_metadata, split_metadata
from features import _items_fingerprint, df_to_items

meta = build_metadata(strict=True)
assert len(meta) == 12162, f"expected 12162 samples, got {len(meta)}"
train_df, val_df, test_df = split_metadata(meta)

# Assert a cache HIT before spending GPU time. The fingerprint hashes absolute
# paths, so a one-character difference in DATA_ROOT between notebooks 01 and 02
# would silently re-extract every feature on the GPU clock.
expected = {}
for aware, key in ((True, "eaaa"), (False, "uniform")):
    expected[f"train_{key}"] = _items_fingerprint(
        plan_augmentation(train_df, emotion_aware=aware))
expected["val"] = _items_fingerprint(df_to_items(val_df))
expected["test"] = _items_fingerprint(df_to_items(test_df))

missing = [f"{d}_{fp}.npz" for d, fp in expected.items()
           if not os.path.exists(f"{config.CACHE_DIR}/{d}_{fp}.npz")]
assert not missing, (
    "CACHE MISS - training would re-extract features on GPU time.\n"
    f"Missing: {missing}\n"
    "DATA_ROOT must be byte-identical to notebook 01's value.")

print("All four caches HIT - no GPU time will be spent on extraction.")

In [ ]:
from train import run_experiment

# Six DISTINCT configurations. `base` doubles as the base-paper reproduction
# and the ablation baseline; `full` doubles as the proposed model and the
# ablation's all-novelties row. Running them separately would waste ~2-3 h.
CONFIGS = [
    ("base", dict(use_afw=False, use_eaaa=False, use_mstc=False, use_cadl=False)),
    ("afw",  dict(use_afw=True,  use_eaaa=False, use_mstc=False, use_cadl=False)),
    ("eaaa", dict(use_afw=False, use_eaaa=True,  use_mstc=False, use_cadl=False)),
    ("mstc", dict(use_afw=False, use_eaaa=False, use_mstc=True,  use_cadl=False)),
    ("cadl", dict(use_afw=False, use_eaaa=False, use_mstc=False, use_cadl=True)),
    ("full", dict(use_afw=True,  use_eaaa=True,  use_mstc=True,  use_cadl=True)),
]

# Kaggle caps sessions at 12 h. Stop cleanly before then; unfinished configs
# resume next session.
MAX_HOURS = 11.0
started = time.time()

for tag, cfg in CONFIGS:
    if os.path.exists(os.path.join(config.RUNS_DIR, tag, "test_metrics.json")):
        print(f"[skip] {tag} already complete")
        continue

    elapsed = (time.time() - started) / 3600
    if elapsed > MAX_HOURS:
        print(f"[stop] {elapsed:.1f} h elapsed - remaining configs deferred "
              f"to the next session")
        break

    print(f"\n{'=' * 72}")
    print(f"[run] {tag}   ({elapsed:.1f} h elapsed)")
    print(f"{'=' * 72}")
    run_experiment(tag=tag, epochs=config.EPOCHS, metadata=meta, **cfg)

In [ ]:
from report import check_acceptance, collect_runs, write_report

df = write_report(config.RUNS_DIR, "/kaggle/working/report")

if df.empty:
    print("no finished runs yet")
else:
    print(df[["tag", "accuracy", "macro_f1", "mcc",
              "params"]].to_string(index=False))
    print()
    for name, ok, detail in check_acceptance(df):
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}: {detail}")

In [ ]:
# /kaggle/working is capped at 20 GB and symlinked corpora would be FOLLOWED
# when the output is saved. Remove the links; keep runs/, report/ and caches.
for name in SOURCES:
    link = os.path.join(DATA_ROOT, name)
    if os.path.islink(link):
        os.unlink(link)

print("output contents:", sorted(os.listdir("/kaggle/working")))

## If the session times out

Nothing is lost. Publish this notebook's output as a dataset, attach it to the
next run alongside the feature cache, and re-run: the runner skips every config
that already has `test_metrics.json` and continues from where it stopped.